# Función de Activación = ReLU, Coeficiente R2, HOLD-Out

In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from tqdm import tqdm
import matplotlib.pyplot as plt
from time import time
from torchmetrics import MeanAbsoluteError, MeanSquaredError, R2Score
from torchmetrics.regression import MeanAbsolutePercentageError
if torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
device

'cpu'

In [2]:
data = pd.read_csv('./structuralparameters-vs-capacities-5000mofs.csv', index_col=False, header=0)
data

,usablegc,usablevc,density,porosity,Ri,SSA,SPV,du,metalw,metala
0,0.2294,0.003505,1.525,0.04,2.22,632,0.026,289.0,19.449,3.704
1,0.1965,0.003082,1.566,0.02,1.77,529,0.013,289.0,19.449,3.704
2,1.3542,0.013027,0.949,0.30,5.70,3024,0.316,217.0,24.053,7.143
3,0.3773,0.004750,1.255,0.04,2.27,1002,0.032,233.0,15.330,2.439
4,0.4118,0.005179,1.253,0.04,2.41,1085,0.032,465.0,15.330,2.439
...,...,...,...,...,...,...,...,...,...,...
4995,0.1503,0.002866,1.905,0.02,2.20,377,0.011,313.0,31.873,5.882
4996,0.2563,0.002970,1.156,0.08,3.92,598,0.069,145.0,9.378,3.279
4997,0.1684,0.002585,1.533,0.06,2.48,479,0.039,201.0,24.548,2.222
4998,0.1743,0.002776,1.590,0.06,2.49,433,0.038,401.0,25.770,2.222


In [3]:
limites_tabla = pd.read_csv('./limits_5000mofs.csv', index_col=False, header=0)
limites_tabla

,Variable,Units,lowlimit,highlimit
0,usablegc,(wt.%),0.000001,15.00
1,usablevc,(kg/L),0.000001,0.06
2,density,(kg/L),0.100000,5.00
3,porosity,(dimensionless),0.000100,0.90
4,R_i,(angstroms),1.000000,40.00
5,ssa,(m^2/g),100.000000,10000.00
6,specificporevolume,(cm^3/g),0.000100,10.00
7,du,(dimensionless),0.000000,4000.00
8,metalw,%,0.000001,90.00
9,metala,%,0.000001,90.00


In [4]:
limit_inputs = np.array(limites_tabla.iloc[0:2, -2:]).T
limit_inputs

array([[1.0e-06, 1.0e-06],
       [1.5e+01, 6.0e-02]])

In [5]:
limit_outputs = np.array(limites_tabla.iloc[2:, -2:]).T
limit_outputs

array([[1.e-01, 1.e-04, 1.e+00, 1.e+02, 1.e-04, 0.e+00, 1.e-06, 1.e-06],
       [5.e+00, 9.e-01, 4.e+01, 1.e+04, 1.e+01, 4.e+03, 9.e+01, 9.e+01]])

In [6]:
escaler_X = MinMaxScaler().fit(limit_inputs)
escaler_y = MinMaxScaler().fit(limit_outputs)

In [7]:
X = torch.tensor(escaler_X.transform(np.array(data.iloc[:, 0:2])), dtype=torch.float32)
X.max(dim=0), X.min(dim=0), X.shape, type(X), X

(torch.return_types.max(
 values=tensor([0.7427, 0.3981]),
 indices=tensor([266,  89])),
 torch.return_types.min(
 values=tensor([0.0006, 0.0040]),
 indices=tensor([1195, 1195])),
 torch.Size([5000, 2]),
 torch.Tensor,
 tensor([[0.0153, 0.0584],
         [0.0131, 0.0514],
         [0.0903, 0.2171],
         ...,
         [0.0112, 0.0431],
         [0.0116, 0.0463],
         [0.0101, 0.0411]]))

In [8]:
y = torch.tensor(escaler_y.fit_transform(np.array(data.iloc[:, 2:])), dtype=torch.float32)
y.max(dim=0), y.min(dim=0), y.shape, type(y),y

(torch.return_types.max(
 values=tensor([1., 1., 1., 1., 1., 1., 1., 1.]),
 indices=tensor([1286, 3284, 3284,  266,  266,   64, 4857, 3611])),
 torch.return_types.min(
 values=tensor([0., 0., 0., 0., 0., 0., 0., 0.]),
 indices=tensor([ 266,   17, 3486, 1195,   17,  114, 4245, 3112])),
 torch.Size([5000, 8]),
 torch.Tensor,
 tensor([[0.3570, 0.0453, 0.0215,  ..., 0.0938, 0.2564, 0.0983],
         [0.3675, 0.0226, 0.0084,  ..., 0.0938, 0.2564, 0.0983],
         [0.2095, 0.3408, 0.1229,  ..., 0.0703, 0.3233, 0.2052],
         ...,
         [0.3590, 0.0681, 0.0291,  ..., 0.0651, 0.3305, 0.0522],
         [0.3736, 0.0681, 0.0294,  ..., 0.1302, 0.3483, 0.0522],
         [0.3813, 0.0453, 0.0288,  ..., 0.1302, 0.3609, 0.0522]]))

In [9]:
# idx_train, idx_test = train_test_split(np.arange(1000), test_size=1/3, random_state=42)

In [10]:
class dataset(Dataset):
  def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
    self.X = X.to(device)
    self.y = y.to(device)
    self.len = self.X.shape[0]
  
  def __getitem__(self, index: int) -> tuple:
    return self.X[index], self.y[index]

  def __len__(self) -> int:
    return self.len

In [11]:
class LinearRegression(nn.Module):
    '''Linear Regression Model
    '''

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int) -> None:
        '''The network has 4 layers 
             - input layer
             - hidden layer
             - hidden layer
             - output layer
        '''
        super(LinearRegression, self).__init__()
        self.act = torch.nn.ReLU()
        self.input_to_hidden = nn.Linear(input_dim, hidden_dim)
        self.hidden_layer_1 = nn.Linear(hidden_dim, 2*hidden_dim)
        self.hidden_layer_2 = nn.Linear(2*hidden_dim, 2*hidden_dim)

        self.hidden_layer_3 = nn.Linear(2*hidden_dim, hidden_dim)
        self.hidden_to_output = nn.Linear(hidden_dim, output_dim)
        self.sigmoide = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # no activation and no softmax at the end
        x = self.act(self.input_to_hidden(x))
        x = self.act(self.hidden_layer_1(x))
        x = self.act(self.hidden_layer_2(x))
        x = self.act(self.hidden_layer_3(x))
        
        x = self.sigmoide(self.hidden_to_output(x))
        return x

In [12]:
CV = KFold(n_splits=10, shuffle=True, random_state=42)

In [13]:
batch_size = 1
input_dim = 2
hidden_layers = 10
output_dim = 8

In [14]:
def evaluate_with_torchmetrics(model, test_loader, device='cpu'):
    """
    Evaluación usando torchmetrics (recomendado para producción)
    """
    model.eval()
    model.to(device)
    
    # Inicializar métricas
    mae_metric = MeanAbsoluteError().to(device)
    mse_metric = MeanSquaredError().to(device)
    r2_metric = R2Score().to(device)
    mape_metric = MeanAbsolutePercentageError().to(device)
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            
            # Actualizar métricas
            mae_metric.update(outputs, targets)
            mse_metric.update(outputs, targets)
            r2_metric.update(outputs, targets)
            mape_metric.update(outputs, targets)
    
    # Calcular resultados finales
    results = {
        'mae': mae_metric.compute().item(),
        'mse': mse_metric.compute().item(),
        'rmse': torch.sqrt(mse_metric.compute()).item(),
        'r2': r2_metric.compute().item(),
        'mape': mape_metric.compute().item()
    }
    
    # Resetear métricas
    mae_metric.reset()
    mse_metric.reset()
    r2_metric.reset()
    mape_metric.reset()
    
    return results

In [15]:
def training(epochs=10, trainloader=None, testloader=None): 
    for epoch in tqdm(range(epochs)):
        model.train()
        for (inputs, labels) in trainloader:
            # forward propagation
            outputs = model(inputs)
            loss = criterion(outputs, labels)
    
            # set optimizer to zero grad 
            # to remove previous epoch gradients
            optimizer.zero_grad()
    
            # backward propagation
            loss.backward()
    
            # optimize
            optimizer.step()          
            
    return evaluate_with_torchmetrics(model=model, test_loader=testloader)

In [16]:
resultados = pd.DataFrame(columns=['mae', 'mse', 'rmse', 'r2', 'mape'])
resultados.index.name= 'cv'
resultados

,mae,mse,rmse,r2,mape
cv,,,,,


In [17]:
for i , (idx_train, idx_test) in enumerate(CV.split(np.arange(1000))):
    print("CV = ", i)
    
    dataset_train = dataset(X[idx_train], y[idx_train])
    dataset_test = dataset(X[idx_test], y[idx_test])

    trainloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
    testloader = DataLoader(dataset_test, batch_size=len(dataset_test))
    
    model = LinearRegression(input_dim, hidden_layers, output_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters())
    rr = training(epochs=100, trainloader=trainloader, testloader=testloader)
    resultados.loc[i] = [rr['mae'], rr['mse'], rr['rmse'], rr['r2'], rr['mape']]  

CV =  0


100%|█████████████████████████████████████████| 100/100 [01:24<00:00,  1.19it/s]


CV =  1


100%|█████████████████████████████████████████| 100/100 [01:24<00:00,  1.18it/s]


CV =  2


...

100%|█████████████████████████████████████████| 100/100 [01:22<00:00,  1.22it/s]


CV =  8


100%|█████████████████████████████████████████| 100/100 [01:21<00:00,  1.23it/s]


CV =  9


100%|█████████████████████████████████████████| 100/100 [01:20<00:00,  1.24it/s]


In [20]:
resultados

,mae,mse,rmse,r2,mape
cv,,,,,
0,0.037426,0.003673,0.060606,0.617377,792.703186
1,0.033688,0.002603,0.051019,0.689314,1423.017456
2,0.036039,0.003152,0.056142,0.683112,256.602081
3,0.036518,0.003167,0.056274,0.602880,650.929504
4,0.033118,0.002535,0.050350,0.702904,670.842834
5,0.032222,0.003061,0.055325,0.643641,503.964661
6,0.031499,0.002423,0.049219,0.696828,589.078613
7,0.033740,0.003223,0.056774,0.680481,943.444519
8,0.036394,0.003856,0.062100,0.692882,584.195068


In [21]:
resultados.mean()

mae       0.034328
mse       0.003024
rmse      0.054831
r2        0.670082
mape    703.319119
dtype: float64